# 07 - Real IBM hardware: projected-kernel features (week 3)
Owner: **Farrukh**

The fourth level of our comparison: **exact -> sampled -> noisy model -> real hardware**.

- **What runs:** the projected kernel's Pauli measurements ($Z$, $X$, $Y$ basis; 3 circuits per image) for **all 120 images of seed 0 at $n=6$ qubits** (360 circuits), plus 2 readout-calibration circuits. The fidelity kernel would need 6,360 circuits per seed, far beyond the Open Plan budget (600 s per 28 days), so it is not run on hardware.
- **Why $n=6$:** the qubit count where the projected kernel's engineered-label result was clearest.
- **Device:** `ibm_kingston`, the machine our noise model (notebook 06) copies; same physical qubits $0,\dots,5$, same transpilation and the same 2048 shots.
- **Settings are not re-tuned:** $\alpha$, scale $s$ and $C$ are the ones chosen for seed 0, $n=6$ in notebook 03; $\gamma$ follows from each level's training features by the same median rule.
- **Safety:** a dry run on the simulated `ibm_kingston` (no quota) comes first. The real job is submitted only when `RUN_HARDWARE = True`, its job ID and raw counts are saved immediately, and the notebook never resubmits once results exist.

In [1]:
import sys, json, time
sys.path.insert(0, "..")
import numpy as np
from qiskit_ibm_runtime.fake_provider import FakeKingston
from src.common import (get_split, make_preprocessor, encoding_circuit, evaluate_precomputed,
                        result_path, RESULTS, SHOTS)
from src.projected import pauli_features_exact, pauli_features_sampled, projected_kernel, median_gamma
from src.baselines import classical_gram
from src.geometry import geometric_difference
from src.hardware import (BACKEND_NAME, build_circuits, transpile_for, dry_run_counts, submit,
                          counts_from_result, features_from_counts, readout_errors,
                          save_counts, load_counts)

## 1. Settings

In [ ]:
SEED, N, LAYERS = 0, 6, 2
RUN_HARDWARE = False          # set to True ONLY for the one real run (section 4)

p = json.loads(result_path("tables", "projected_v2", SEED, N, "json").read_text())
b = json.loads(result_path("tables", "baselines_v2", SEED, N, "json").read_text())
ALPHA, GSCALE, C = p["best"]["alpha"], p["best"]["gscale"], p["best"]["C"]
KINDS = {"SVM RBF (tuned)": "rbf", "SVM polynomial (tuned)": "poly", "SVM linear (tuned)": "linear"}
print(f"seed {SEED}, n={N}: alpha={ALPHA}, gamma scale={GSCALE}, C={C} (from notebook 03)")
print(f"best classical C* = {b['c_star']}, test accuracy {b['results'][b['c_star']]['acc']:.3f}")

seed 0, n=6: alpha=2.0, gamma scale=2.0, C=1 (from notebook 03)
best classical C* = SVM polynomial (tuned), test accuracy 0.925


## 2. Circuits

In [3]:
Xtr, ytr, Xte, yte = get_split(SEED)
tf = make_preprocessor(N, ALPHA)(Xtr)                   # fitted on training images only
Atr, Ate = tf(Xtr), tf(Xte)
A_all = np.vstack([Atr, Ate])                           # 80 train + 40 test images
n_tr, n_all = len(Atr), len(A_all)
enc = encoding_circuit(N, LAYERS)
circs = build_circuits(enc, A_all)
print(f"{len(circs)} circuits = {n_all} images x 3 bases + 2 calibration, {SHOTS} shots each")

fake = FakeKingston()
tc_fake = transpile_for(fake, circs, N)
ops = tc_fake[2].count_ops()
print(f"transpiled basis circuit: depth {tc_fake[2].depth()}, {ops.get('cz', 0)} CZ, "
      f"physical qubits {list(tc_fake[2].layout.final_index_layout())}")

362 circuits = 120 images x 3 bases + 2 calibration, 2048 shots each
transpiled basis circuit: depth 33, 10 CZ, physical qubits [0, 1, 2, 3, 4, 5]


## 3. Dry run on the simulated `ibm_kingston` (no quota used)

In [4]:
def split_counts(counts):
    return counts[:3 * n_all], counts[3 * n_all], counts[3 * n_all + 1]

def analyse(F_all, label):
    """Projected kernel from features of all 120 images: test accuracy, Delta A, g vs C*."""
    Ftr, Fte = F_all[:n_tr], F_all[n_tr:]
    gam = median_gamma(Ftr, GSCALE)
    K, Kt = projected_kernel(Ftr, gamma=gam), projected_kernel(Fte, Ftr, gamma=gam)
    res = evaluate_precomputed(K, Kt, ytr, yte, C)
    Fc = make_preprocessor(N, 1.0)(Xtr)(Xtr)            # classical features, as notebook 01
    KC = classical_gram(KINDS[b["c_star"]], b["results"][b["c_star"]]["params"], Fc)
    acc_c = b["results"][b["c_star"]]["acc"]
    return dict(level=label, acc=res["acc"], auc=res["auc"], delta_A=res["acc"] - acc_c,
                g=geometric_difference(KC, K)[0],
                feat_err=float(np.abs(F_all - F_exact).mean()),
                bloch=float(np.linalg.norm(F_all, axis=-1).mean()),
                mean_Y=float(F_all[..., 1].mean()))

F_exact = pauli_features_exact(enc, A_all)
dry_path = result_path("hardware", "dryrun_counts", SEED, N, "json")
t0 = time.time()
dry_counts = dry_run_counts(fake, tc_fake, seed=SEED)
save_counts(dry_path, "local-dry-run", "FakeKingston", dry_counts, dict(seed=SEED, n=N, shots=SHOTS))
c_basis, c0, c1 = split_counts(dry_counts)
F_dry = features_from_counts(c_basis, n_all, N)
p10, p01 = readout_errors(c0, c1, N)
r_dry = analyse(F_dry, "dry run (FakeKingston)")
print(f"dry run done in {time.time() - t0:.0f} s, saved {dry_path.name}")
print(f"test acc {r_dry['acc']:.3f}, Delta A {r_dry['delta_A']:+.3f}, g {r_dry['g']:.2f}, "
      f"feature error vs exact {r_dry['feat_err']:.3f}, mean <Y> {r_dry['mean_Y']:+.4f}")
print("readout error p(1|0) per qubit:", np.round(p10, 3))
print("readout error p(0|1) per qubit:", np.round(p01, 3))

dry run done in 17 s, saved dryrun_counts_seed0_q6.json
test acc 0.800, Delta A -0.125, g 3.38, feature error vs exact 0.029, mean <Y> +0.0003
readout error p(1|0) per qubit: [0.011 0.166 0.009 0.008 0.011 0.007]
readout error p(0|1) per qubit: [0.011 0.183 0.01  0.007 0.011 0.007]


## 4. Real hardware run on `ibm_kingston`

**Run this cell once.** With `RUN_HARDWARE = False` it does nothing. Set it to `True` in section 1, rerun section 1 and this cell. The job ID is saved the moment the job is submitted; if the kernel or internet drops, rerunning this cell fetches the finished job by its ID instead of submitting a new one. Once the counts file exists, the cell only loads it.

In [8]:
from qiskit_ibm_runtime import QiskitRuntimeService

hw_path = result_path("hardware", "hw_counts", SEED, N, "json")
id_path = result_path("hardware", "hw_job_id", SEED, N, "txt")

if hw_path.exists():
    hw_id, hw_backend, hw_meta, hw_counts = load_counts(hw_path)
    print(f"loaded saved hardware counts (job {hw_id}, {hw_backend}) - nothing submitted")
elif id_path.exists():
    hw_id = id_path.read_text().strip()
    job = QiskitRuntimeService().job(hw_id)
    print(f"fetching existing job {hw_id}, status: {job.status()}")
    hw_counts = counts_from_result(job.result(), len(circs))
    save_counts(hw_path, hw_id, BACKEND_NAME, hw_counts, dict(seed=SEED, n=N, shots=SHOTS))
    print("saved", hw_path.name)
elif RUN_HARDWARE:
    backend = QiskitRuntimeService().backend(BACKEND_NAME)
    tc_hw = transpile_for(backend, circs, N)
    print(f"submitting {len(tc_hw)} circuits x {SHOTS} shots to {backend.name} ...")
    job = submit(backend, tc_hw)
    hw_id = job.job_id()
    id_path.write_text(hw_id)                          # saved before waiting
    print(f"job ID {hw_id} saved to {id_path.name}; waiting for results (queue + run) ...")
    t0 = time.time()
    hw_counts = counts_from_result(job.result(), len(circs))
    save_counts(hw_path, hw_id, BACKEND_NAME, hw_counts, dict(seed=SEED, n=N, shots=SHOTS))
    print(f"done after {time.time() - t0:.0f} s; saved {hw_path.name}")
    try:
        print("quantum time used by this job (s):", job.usage())
    except Exception as e:
        print("usage not reported:", e)
else:
    print("RUN_HARDWARE is False: nothing submitted. Check the dry run above first.")

submitting 362 circuits x 2048 shots to ibm_kingston ...
job ID dc273udhdvas73bload0 saved to hw_job_id_seed0_q6.txt; waiting for results (queue + run) ...
done after 247 s; saved hw_counts_seed0_q6.json
quantum time used by this job (s): 197


## 5. Four-level comparison (seed 0, $n=6$)

In [9]:
levels = [analyse(F_exact, "exact")]
F_s = np.vstack([pauli_features_sampled(enc, Atr, shots=SHOTS, seed=SEED),
                 pauli_features_sampled(enc, Ate, shots=SHOTS, seed=SEED + 1)])
levels.append(analyse(F_s, "sampled (2048 shots)"))
F_n = np.vstack([np.load(result_path("features", "pauli_noisy_train", SEED, N, "npy")),
                 np.load(result_path("features", "pauli_noisy_test", SEED, N, "npy"))])
levels.append(analyse(F_n, "noisy model (notebook 06)"))
levels.append(r_dry)
if hw_path.exists():
    _, _, _, hw_counts = load_counts(hw_path)
    c_basis, c0, c1 = split_counts(hw_counts)
    F_hw = features_from_counts(c_basis, n_all, N)
    hw_p10, hw_p01 = readout_errors(c0, c1, N)
    levels.append(analyse(F_hw, f"real hardware ({BACKEND_NAME})"))
    np.save(result_path("features", "pauli_hardware_all", SEED, N, "npy"), F_hw)

acc_c = b["results"][b["c_star"]]["acc"]
print(f"seed {SEED}, n={N}; best classical C* test accuracy {acc_c:.3f}; all levels use the same gamma rule")
print(f"{'level':<30}{'test acc':>9}{'Delta A':>9}{'g':>7}{'feat err':>10}{'Bloch':>8}{'<Y>':>9}")
for r in levels:
    print(f"{r['level']:<30}{r['acc']:>9.3f}{r['delta_A']:>+9.3f}{r['g']:>7.2f}"
          f"{r['feat_err']:>10.3f}{r['bloch']:>8.3f}{r['mean_Y']:>+9.4f}")
if hw_path.exists():
    print("\nhardware readout error p(1|0):", np.round(hw_p10, 3))
    print("hardware readout error p(0|1):", np.round(hw_p01, 3))
    out = dict(seed=SEED, n_qubits=N, backend=BACKEND_NAME, shots=SHOTS,
               settings=dict(alpha=ALPHA, gscale=GSCALE, C=C), c_star=b["c_star"], acc_c_star=acc_c,
               levels=levels, readout_p10=hw_p10.tolist(), readout_p01=hw_p01.tolist())
    result_path("tables", "hardware_projected", SEED, N, "json").write_text(json.dumps(out, indent=2))
    print("saved hardware_projected json")
else:
    print("\n(no hardware results yet - run section 4)")

seed 0, n=6; best classical C* test accuracy 0.925; all levels use the same gamma rule
level                          test acc  Delta A      g  feat err   Bloch      <Y>
exact                             0.800   -0.125   3.37     0.000   0.521  +0.0000
sampled (2048 shots)              0.775   -0.150   3.38     0.016   0.522  +0.0005
noisy model (notebook 06)         0.775   -0.150   3.38     0.030   0.480  +0.0006
dry run (FakeKingston)            0.800   -0.125   3.38     0.029   0.479  +0.0003
real hardware (ibm_kingston)      0.825   -0.100   3.36     0.023   0.503  +0.0046

hardware readout error p(1|0): [0.011 0.    0.007 0.003 0.005 0.003]
hardware readout error p(0|1): [0.02  0.019 0.013 0.016 0.012 0.011]
saved hardware_projected json
